# 02 Preprocessing and Feature Engineering

This notebook prepares the first customer-level feature table for future segmentation. The table uses `customer_info` as the base because it contains the full customer population. Basket-derived features are added with a left join so customers without sampled baskets remain in the table.

This phase does not train clustering models and does not create final clustering outputs.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if hasattr(obj, "to_string"):
            print(obj.to_string())
        else:
            print(obj)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "customer_info.csv").exists() and (candidate / "customer_basket.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate project raw datasets from the current path.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REFERENCE_DATE = pd.Timestamp.today().normalize()
print(f"Project root: {PROJECT_ROOT}")
print(f"Reference date for age and tenure: {REFERENCE_DATE.date()}")

In [ ]:
from src.data_loading import load_datasets
from src.data_audit import missing_values
from src.features import build_customer_feature_table

## Feature Engineering Decisions

- `customer_info` is the base table, so the feature table must preserve every customer.
- `customer_id` is retained only as the row key. Raw identifiers such as `customer_name` and `loyalty_card_number` are not modeling features.
- Birthdate is converted to `customer_age`; invalid or missing ages are flagged before numeric imputation.
- First transaction year is converted to `customer_tenure_years`; future or implausible years are flagged and handled before imputation.
- Loyalty card number is converted to `has_loyalty_card` and `loyalty_card_missing`.
- Promotion percentages are clipped to `[0, 1]` for `promotion_pct_clean`, with suspicious values preserved in a flag.
- Missing numeric customer values are median-imputed with missingness indicators when a column has missing values.
- Basket features are computed per customer and left-joined to the full customer base. Customers without sampled baskets receive zero defaults for basket counts and sizes.

## Load Raw Data

In [ ]:
customer_info, customer_basket = load_datasets(PROJECT_ROOT)

customers_before = customer_info["customer_id"].nunique()
print(f"customer_info rows: {len(customer_info):,}")
print(f"unique customers before feature engineering: {customers_before:,}")
print(f"customer_basket rows: {len(customer_basket):,}")

## Build Customer-Level Feature Table

In [ ]:
feature_table, metadata = build_customer_feature_table(
    customer_info,
    customer_basket,
    reference_date=REFERENCE_DATE,
)

customers_after = feature_table["customer_id"].nunique()
print(f"feature table shape: {feature_table.shape}")
print(f"unique customers after feature engineering: {customers_after:,}")
print(f"basket parse errors: {metadata['basket_parse_errors']:,}")
print(f"customers without sampled baskets: {metadata['customers_without_baskets']:,}")
display(feature_table.head())

## Row Preservation Checks

In [ ]:
assert len(feature_table) == len(customer_info), "Feature table row count changed."
assert feature_table["customer_id"].is_unique, "customer_id is not unique after feature engineering."
assert set(feature_table["customer_id"]) == set(customer_info["customer_id"]), "Customer IDs do not match customer_info."
assert metadata["basket_parse_errors"] == 0, "Some basket rows could not be parsed."

no_basket = feature_table["basket_count"] == 0
assert (feature_table.loc[no_basket, "avg_basket_size"] == 0).all(), "No-basket customers must have avg_basket_size = 0."
assert (feature_table.loc[no_basket, "unique_basket_products"] == 0).all(), "No-basket customers must have unique_basket_products = 0."

print("PASS: every customer from customer_info is present exactly once.")
print(f"Customers before: {customers_before:,}")
print(f"Customers after: {customers_after:,}")
print(f"Customers without baskets retained: {int(no_basket.sum()):,}")

## Missing Values After Preprocessing

In [ ]:
post_missing = missing_values(feature_table)
display(post_missing if not post_missing.empty else pd.DataFrame({"message": ["No missing values remain after preprocessing."]}))
print(f"Total missing values after preprocessing: {int(feature_table.isna().sum().sum()):,}")

## Engineered Feature Distributions

In [ ]:
selected_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "total_children_home",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

distribution = feature_table[selected_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T
display(distribution)

## Key Feature Groups Created

In [ ]:
feature_groups = {
    "demographic": ["customer_age", "gender_female", "gender_male", "gender_unknown"],
    "family": ["kids_home", "teens_home", "total_children_home", "has_children_home"],
    "spend": [column for column in feature_table.columns if column.startswith("spend_share_")] + ["total_lifetime_spend"],
    "loyalty_complaints": ["has_loyalty_card", "loyalty_card_missing", "number_complaints"],
    "promotion": ["promotion_pct_clean", "promotion_pct_suspicious", "promotion_pct_missing"],
    "basket": ["basket_count", "avg_basket_size", "unique_basket_products", "has_sampled_basket"],
}

for group, columns in feature_groups.items():
    present = [column for column in columns if column in feature_table.columns]
    print(f"{group}: {len(present)} features")
    print(present)
    print()

## Customers With Baskets vs Without Baskets

In [ ]:
comparison_features = [
    "customer_age",
    "customer_tenure_years",
    "total_lifetime_spend",
    "promotion_pct_clean",
    "has_loyalty_card",
    "number_complaints",
    "basket_count",
    "avg_basket_size",
    "unique_basket_products",
]

basket_comparison = (
    feature_table.assign(basket_status=feature_table["has_sampled_basket"].map({1: "with baskets", 0: "without baskets"}))
    .groupby("basket_status")[comparison_features]
    .agg(["count", "mean", "median"])
)
display(basket_comparison)

## Data Quality Flags to Carry Forward

In [ ]:
quality_flag_columns = [
    column
    for column in feature_table.columns
    if column.endswith("_was_missing")
    or column.endswith("_missing")
    or column.endswith("_missing_or_invalid")
    or column.endswith("_suspicious")
    or column.endswith("_parse_failed")
]

quality_flags = (
    feature_table[quality_flag_columns]
    .sum()
    .sort_values(ascending=False)
    .rename("flagged_rows")
    .reset_index()
    .rename(columns={"index": "feature"})
)
display(quality_flags[quality_flags["flagged_rows"] > 0])

## Phase Summary

In [ ]:
summary = {
    "customers_before": customers_before,
    "customers_after": customers_after,
    "feature_columns_including_customer_id": feature_table.shape[1],
    "customers_without_baskets": int((feature_table["basket_count"] == 0).sum()),
    "total_missing_after_preprocessing": int(feature_table.isna().sum().sum()),
    "clustering_performed": False,
}
display(pd.DataFrame([summary]))
print("Feature engineering complete. No clustering was performed and no final output file was created.")